In [80]:
!pip install transformers torch

In [89]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [91]:

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.resize_token_embeddings(len(tokenizer))

print("Model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


In [92]:
def get_ai_response(chat_history):
    # Convert chat history to model-specific format
    inputs = tokenizer.apply_chat_template(
        chat_history,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Generate response
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id
    )

    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Post-processing for clean output
    response = response.split("\n")[0].split("###")[0].strip()
    if "." in response:
        response = response.split(".")[0] + "."

    return response if len(response) > 5 else "I am here to help. Please ask your question."

In [93]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

# Store conversation history with a system prompt
chat_history = [
    {"role": "system", "content": "You are a helpful AI assistant. Give short, clear, and accurate answers."}
]

while True:
    user_input = input("You: ").strip()

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    if not user_input:
        continue

    # Add user message to history
    chat_history.append({"role": "user", "content": user_input})

    # Get response from the model
    try:
        bot_reply = get_ai_response(chat_history)
        print(f"Chatbot: {bot_reply}")

        # Save bot reply to history for continuous conversation
        chat_history.append({"role": "assistant", "content": bot_reply})
    except Exception as e:
        print(f"Chatbot: I encountered an error. Please try again.")

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: hello
Chatbot: Hello! How can I assist you today?
You: what is artificial intelligence
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans.
You: who created python
Chatbot: Python was created by Guido van Rossum, who developed it as part of the Python Software Foundation.
You: thank you!
Chatbot: You're welcome! If you have any more questions, feel free to ask.
You: exit
Chatbot: Goodbye!
